# Tutorial: Affiliation Pipeline

Dieses Notebook ist gleichzeitig **Anleitung, Beispiel und Referenz** für die Affiliation-Pipeline.

Es zeigt den normalen Ablauf von einer Roh-CSV bis zum ROR-Ergebnis:

`CSV → Spalten vorbereiten → Affiliations aufteilen → Orte entfernen → ROR abfragen → Ergebnisse prüfen → CSV speichern`

Wichtig:

- Die ursprünglichen Rohspalten werden nicht überschrieben.
- Die ROR API bekommt `affiliation_part_raw`, also den möglichst ursprünglichen Affiliation-Text.
- `affiliation_clean` wird für die bereinigte Organisationsstruktur verwendet.
- `ror_original` ist optional und wird nicht als fehlerfreie Ground Truth behandelt.
- `review` bedeutet nur, dass ein Fall bewusst genauer angesehen werden sollte.
- `unmatched` ist nicht automatisch ein Fehler. Viele Firmen, GmbHs oder kleine Einrichtungen haben keinen ROR-Eintrag.
- Eine vollständig automatische Zuordnung aller denkbaren Affiliationen ist nicht das Ziel. Die Pipeline soll die manuelle Arbeit stark reduzieren und problematische Fälle sichtbar machen.

## 0. Erwartete Projektstruktur

Standardmäßig wird folgende Struktur erwartet:

```text
affiliation-pipeline/
├── affiliation_pipeline/
│   ├── __init__.py
│   ├── preprocessing.py
│   ├── locations.py
│   └── ror_api.py
├── data/
│   ├── affilation_test.csv
│   ├── cities500/
│   └── alternateNamesV2/
├── jupyter/
│   └── tutorial_affiliation_pipeline_v1.ipynb
├── .env
└── requirements.txt
```

Die Datei `affilation_test.csv` ist hier nur der **Default für das Tutorial**.

Für eigene Daten:

1. CSV in den Ordner `data/` kopieren.
2. In der nächsten Konfigurationszelle nur `DATA_FILENAME` ändern.
3. Falls die Affiliation-Spalte anders heißt, zusätzlich `SOURCE_AFFILIATION_COLUMN` ändern.

Die GeoNames-Daten werden lokal benötigt und gehören normalerweise **nicht** ins Git-Repository.

## 1. Zentrale Konfiguration

Diese Zelle ist normalerweise die einzige Stelle, die für einen neuen Datensatz angepasst werden muss.

### Zwingend

- `DATA_FILENAME`: Name der Eingabe-CSV im `data/`-Ordner.
- `SOURCE_AFFILIATION_COLUMN`: Spalte mit dem ursprünglichen Affiliation-Text.

### Optional

- `ORIGINAL_ROR_COLUMN`: bestehende ROR-Spalte. Auf `None` setzen, wenn es keine gibt.
- `SPLIT_AFFILIATIONS`: nur `True`, wenn mehrere Affiliations innerhalb einer Zelle durch einen festen Delimiter getrennt sind.
- `AFFILIATION_DELIMITER`: in unseren Testdaten `;`.
- `INCLUDE_LOCATION_DIAGNOSTICS`: zusätzliche Diagnosefelder für Stadt-/Ländererkennung.
- `RUN_ROR_API`: kann für reine Preprocessing-Tests auf `False` gesetzt werden.

In [ ]:
DATA_FILENAME = "affilation_test.csv"

SOURCE_AFFILIATION_COLUMN = "Unnamed: 1"
STANDARD_AFFILIATION_COLUMN = "affiliation"

ORIGINAL_ROR_COLUMN = "Unnamed: 2"

SPLIT_AFFILIATIONS = True
AFFILIATION_DELIMITER = ";"

INCLUDE_LOCATION_DIAGNOSTICS = True
REMOVE_ADDRESSES = True

RUN_ROR_API = True

OUTPUT_FILENAME = "affiliation_result.csv"

RANDOM_STATE = 42

## 2. Imports und Projektpfad

Das Notebook kann entweder direkt im Projektordner oder im Unterordner `jupyter/` liegen.

`PROJECT_ROOT` wird deshalb automatisch bestimmt.

Die wichtigsten Pipeline-Funktionen sind:

- `load_csv()` – CSV laden.
- `preview_dataframe()` – Spalten und zufällige Beispielzeilen ansehen.
- `rename_columns()` – Spalten umbenennen, ohne den Code an einer bestimmten Quelldatei festzumachen.
- `split_column()` – eine Spalte entweder in neue Zeilen oder neue Spalten aufteilen.
- `load_location_resources()` – Länder- und GeoNames-Ressourcen laden.
- `process_locations()` – Stadt, Land, PLZ und Adresse extrahieren und `affiliation_clean` erzeugen.
- `norm_country()` / `norm_city()` – einzelne Länder- bzw. Städtenamen normalisieren.
- `RORAffiliationProcessor` – ROR-Abfrage, Cache, Fallback, Status und strukturierte Clean-Spalten.
- `save_csv()` – Ergebnis speichern.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "jupyter":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from affiliation_pipeline.preprocessing import (
    load_csv,
    preview_dataframe,
    rename_columns,
    save_csv,
    split_column,
)

from affiliation_pipeline.locations import (
    load_location_resources,
    norm_city,
    norm_country,
    process_locations,
)

from affiliation_pipeline.ror_api import RORAffiliationProcessor

DATA_FILE = PROJECT_ROOT / "data" / DATA_FILENAME
CACHE_FILE = PROJECT_ROOT / "data" / "ror_api_cache.json"
CHECKPOINT_FILE = PROJECT_ROOT / "data" / "ror_api_checkpoint.csv"

print("Projekt:", PROJECT_ROOT)
print("Eingabedatei:", DATA_FILE)

## 3. CSV laden – `load_csv()`

`load_csv()` liest die Datei als DataFrame ein.

Für dieses Projekt ist `;` der übliche **CSV-Feldtrenner**.

Das ist etwas anderes als ein Semikolon, das innerhalb eines Affiliation-Feldes mehrere Affiliations trennt. Wenn ein Feld selbst Semikolons enthält, muss die CSV korrekt gequotet sein.

Standardanwendung:

```python
df = load_csv(DATA_FILE)
```

Falls eine fremde CSV z. B. `,` als Feldtrenner verwendet, kann der entsprechende pandas-Parameter weitergegeben werden, z. B.:

```python
df = load_csv(DATA_FILE, sep=",")
```

In [ ]:
df = load_csv(DATA_FILE)

print(f"Geladene Zeilen: {len(df):,}")
print(f"Spalten: {len(df.columns)}")

preview_dataframe(
    df,
    n=5,
    random_state=RANDOM_STATE,
)

## 4. Spaltennamen vereinheitlichen – `rename_columns()`

Die Pipeline soll intern mit einer einheitlichen Spalte `affiliation` arbeiten.

Unsere Testdatei verwendet absichtlich `affilation`. Deshalb wird sie hier umbenannt.

`rename_columns()` bekommt ein Mapping:

```python
{
    "alter_name": "neuer_name"
}
```

Mehrere Spalten können gleichzeitig umbenannt werden.

Diese Stufe ist **optional**, wenn die Quelldatei bereits die gewünschten Namen besitzt.

In [ ]:
rename_map = {}

if (
    SOURCE_AFFILIATION_COLUMN != STANDARD_AFFILIATION_COLUMN
    and SOURCE_AFFILIATION_COLUMN in df.columns
):
    rename_map[SOURCE_AFFILIATION_COLUMN] = STANDARD_AFFILIATION_COLUMN

if rename_map:
    df = rename_columns(df, rename_map)

if STANDARD_AFFILIATION_COLUMN not in df.columns:
    raise KeyError(
        f"Die Affiliation-Spalte {STANDARD_AFFILIATION_COLUMN!r} wurde nicht gefunden."
    )

print("Verwendete Affiliation-Spalte:", STANDARD_AFFILIATION_COLUMN)

## 5. Mehrere Affiliations in einer Zelle – `split_column()`

In manchen Quelldaten enthält eine Zeile mehrere Affiliations, z. B.:

```text
University A;Company B;Institute C
```

Dann sollte jede Affiliation eine eigene Zeile bekommen.

### Variante A: in Zeilen aufteilen

Für den ROR-Workflow ist das normalerweise die richtige Variante:

```python
split_column(
    df,
    column="affiliation",
    delimiter=";",
    into="rows",
    output_column="affiliation_part_raw",
)
```

Dabei:

- bleibt `affiliation` unverändert,
- wird `affiliation_part_raw` neu erzeugt,
- werden die übrigen Spalten für jede neue Zeile mitkopiert,
- kann `original_row_id` gespeichert werden, damit die Herkunft nachvollziehbar bleibt.

### Variante B: in Spalten aufteilen

Für andere Datenbereinigungsaufgaben kann derselbe Helper auch horizontal splitten:

```python
split_column(
    df,
    column="example",
    delimiter="|||",
    into="columns",
    output_prefix="example_part",
)
```

Dann entstehen automatisch `example_part_1`, `example_part_2`, ... bis zur maximal tatsächlich benötigten Anzahl.

Für die ROR-Pipeline wird im Folgenden **Variante A** verwendet.

In [ ]:
if SPLIT_AFFILIATIONS:
    df_parts = split_column(
        df,
        column=STANDARD_AFFILIATION_COLUMN,
        delimiter=AFFILIATION_DELIMITER,
        into="rows",
        output_column="affiliation_part_raw",
        row_id_column="original_row_id",
        drop_empty=True,
    )
else:
    df_parts = df.copy()
    df_parts["affiliation_part_raw"] = (
        df_parts[STANDARD_AFFILIATION_COLUMN]
        .astype("string")
        .str.strip()
    )

print(f"Zeilen vorher: {len(df):,}")
print(f"Zeilen nach Vorbereitung: {len(df_parts):,}")

display(
    df_parts[
        [
            column
            for column in [
                "original_row_id",
                STANDARD_AFFILIATION_COLUMN,
                "affiliation_part_raw",
            ]
            if column in df_parts.columns
        ]
    ].head(10)
)

## 6. GeoNames- und Länderressourcen laden – `load_location_resources()`

Diese Funktion baut die Referenzen für Länder und Städte auf.

- [`cities500.zip`](https://download.geonames.org/export/dump/cities500.zip)  
  Enthält Orte mit mindestens 500 Einwohnern sowie weitere relevante geografische Einträge.

- [`alternateNamesV2.zip`](https://download.geonames.org/export/dump/alternateNamesV2.zip)  
  Enthält alternative Namen, Übersetzungen und Schreibweisen für GeoNames-Einträge.

Alle GeoNames-Download-Dateien sind auch im offiziellen Download-Verzeichnis verfügbar:

<https://download.geonames.org/export/dump/>

Nach dem Download müssen die ZIP-Dateien entpackt und in folgende Ordner gelegt werden:

```text
data/
├── cities500/
│   └── cities500.txt
└── alternateNamesV2/
    └── alternateNamesV2.txt

Die GeoNames-Dateien werden nicht im Git-Repository gespeichert und müssen daher lokal heruntergeladen werden.

Standardmäßig werden die Daten aus dem Projektordner `data/` geladen.

Wichtige optionale Parameter:

- `data_dir=` – anderer Basisordner für GeoNames-Dateien.
- `cities_path=` – konkreter Pfad zur `cities500`-Datei.
- `alternate_names_path=` – konkreter Pfad zu `alternateNamesV2`.
- `language_codes_path=` – optionaler Pfad zu GeoNames-Sprachcodes.
- `locales=("en", "de")` – Sprachvarianten für Ländernamen.
- `additional_country_aliases={...}` – eigene zusätzliche Länderaliase.
- `show_progress=True` – Fortschrittsanzeige beim Laden.
- `alternate_names_chunksize=` – Chunk-Größe beim Einlesen großer Alternate-Names-Dateien.

Für den normalen Projektaufbau reicht:

```python
resources = load_location_resources()
```

Das Laden kann etwas dauern, weil `alternateNamesV2` groß ist.

In [ ]:
resources = load_location_resources(
    show_progress=True,
)

## 7. Einzelne Länder oder Städte testen – `norm_country()` und `norm_city()`

Diese beiden Funktionen sind **nicht zwingend für die komplette Pipeline**. Sie sind kleine Hilfsfunktionen für Tests oder manuelle Prüfungen.

Beispiele:

```python
norm_country("Austria", resources)
norm_country("AT", resources)

norm_city("Vienna", resources)
norm_city("Wien", resources)
```

Die Erkennung ist bewusst konservativ. Ein Name soll nicht nur deshalb aus einer Organisation entfernt werden, weil er zufällig auch ein Ortsname ist.

In [ ]:
print("Country:", norm_country("Austria", resources))
print("Country Code:", norm_country("AT", resources))
print("City Vienna:", norm_city("Vienna", resources))
print("City Wien:", norm_city("Wien", resources))

## 8. Location Cleaning – `process_locations()`

Das ist die zentrale Location-Funktion.

Sie bekommt:

### Zwingend

- das DataFrame,
- `column=` – die Spalte, die bereinigt werden soll,
- `resources=` – die vorher geladenen Länder-/GeoNames-Ressourcen.

### Optional

- `output_column="affiliation_clean"` – Name der bereinigten Affiliation-Spalte.
- `output_columns=[...]` – welche extrahierten Location-Spalten wirklich angelegt werden sollen.
- `remove_addresses=True` – erkannte Adresssegmente aus `affiliation_clean` entfernen.
- `include_diagnostics=True` – zusätzliche Status-/Diagnosespalten.
- `show_progress=True` – `tqdm`-Fortschrittsanzeige.

Mögliche reguläre Location-Ausgaben sind:

- `address_raw`
- `postcode_raw`
- `city_raw`
- `city_norm`
- `country_raw`
- `country_norm`

Bei `include_diagnostics=True` kommen zusätzlich Diagnosefelder dazu, z. B. `country_iso2`, `country_status`, `city_status`, `city_entity_status`, `city_match_type` und `city_source_segment`.

Wichtig: `affiliation_part_raw` bleibt unverändert. `affiliation_clean` ist eine **neue** Spalte.

In [ ]:
location_output_columns = [
    "address_raw",
    "postcode_raw",
    "city_raw",
    "city_norm",
    "country_raw",
    "country_norm",
]

df_clean = process_locations(
    df_parts,
    column="affiliation_part_raw",
    resources=resources,
    output_column="affiliation_clean",
    output_columns=location_output_columns,
    remove_addresses=REMOVE_ADDRESSES,
    include_diagnostics=INCLUDE_LOCATION_DIAGNOSTICS,
    show_progress=True,
)

preview_columns = [
    column
    for column in [
        "id",
        "affiliation_part_raw",
        "affiliation_clean",
        "address_raw",
        "postcode_raw",
        "city_raw",
        "city_norm",
        "country_raw",
        "country_norm",
        "country_iso2",
        "country_status",
        "city_status",
    ]
    if column in df_clean.columns
]

display(
    df_clean[preview_columns].sample(
        n=min(15, len(df_clean)),
        random_state=RANDOM_STATE,
    )
)

## 9. Optional: nur Preprocessing verwenden, ohne ROR API

Die Pipeline kann auch ohne ROR verwendet werden.

Wenn nur Location Cleaning benötigt wird:

```python
RUN_ROR_API = False
```

Dann kann `df_clean` direkt gespeichert oder weiterverarbeitet werden.

Die folgenden API-Zellen werden in diesem Tutorial nur ausgeführt, wenn `RUN_ROR_API = True` gesetzt ist.

## 10. ROR-Konfiguration und `.env`

Der ROR-Processor verwendet standardmäßig:

- `affiliation_part_raw` als API-Eingabe,
- `affiliation_clean` für die spätere Strukturierung,
- `country_iso2` / `country_norm` als zusätzliche Information für Exact-Fallbacks.

### `.env`

Falls ein ROR Client-ID verwendet wird:

```text
ROR_CLIENT_ID=...
```

Die `.env` bleibt lokal und sollte nicht nach GitHub committed werden.

Ohne Client-ID funktioniert die Pipeline ebenfalls, verwendet aber einen deutlich größeren Request-Abstand.

### Cache

`ror_api_cache.json` speichert bereits ausgeführte Requests. Wiederholte identische Anfragen müssen dadurch nicht noch einmal an die API geschickt werden.

### Checkpoint

`ror_api_checkpoint.csv` wird während längerer Läufe regelmäßig gespeichert. Das ist nur eine Sicherheitskopie des Zwischenstands.

Cache und Checkpoint dürfen gelöscht werden, wenn bewusst ein komplett frischer Test gestartet werden soll.

## 11. `RORAffiliationProcessor` erstellen

### Zwingende bzw. zentrale Parameter

- `api_input_col` – Text, der an ROR geschickt wird. In unserem Workflow: `affiliation_part_raw`.
- `clean_col` – bereinigter Text für die spätere Strukturierung. In unserem Workflow: `affiliation_clean`.

### Häufig verwendete optionale Parameter

- `original_ror_col` – bestehende ROR-Spalte oder `None`.
- `country_code_col` – ISO-2-Länderspalte, normalerweise `country_iso2`.
- `country_name_col` – normalisiertes Land, normalerweise `country_norm`.
- `clean_output_prefix` – Präfix für `affiliation_clean_1`, `_2`, ...
- `output_ror_col` – standardmäßig `ror_after_api`.
- `cache_file` – eigener Cache-Pfad.
- `max_fallback_queries` – maximale Zahl zusätzlicher Exact-Fallback-Abfragen pro problematischer Zeile.
- `timeout` – Request-Timeout in Sekunden.
- `max_retries` – Anzahl Wiederholungsversuche.
- `min_request_interval` – Request-Abstand manuell überschreiben.
- `checkpoint_every` – nach wie vielen verarbeiteten Zeilen Cache/Checkpoint geschrieben werden.
- `chosen_locator_threshold` – interne Plausibilitätsgrenze für `chosen=true`-Treffer.

Normalerweise sollten die Defaults nicht ohne konkreten Grund verändert werden.

- `min_request_interval` – Request-Abstand manuell überschreiben.  
  Aktuell erlaubt ROR bis zu 2.000 Requests pro 5 Minuten. Ein Wert von etwa `0.25` Sekunden ist dafür ein konservativer praktischer Wert.  
  Sobald die angekündigten Client-ID-Limits aktiviert werden, gilt: mit gültiger Client ID weiterhin bis zu 2.000 Requests pro 5 Minuten; ohne Client ID nur 50 Requests pro 5 Minuten, wofür etwa `6.5` Sekunden Abstand sinnvoll sind.

In [ ]:
original_ror_col = (
    ORIGINAL_ROR_COLUMN
    if ORIGINAL_ROR_COLUMN
    and ORIGINAL_ROR_COLUMN in df_clean.columns
    else None
)

processor = RORAffiliationProcessor(
    api_input_col="affiliation_part_raw",
    clean_col="affiliation_clean",
    original_ror_col=original_ror_col,
    country_code_col=(
        "country_iso2"
        if "country_iso2" in df_clean.columns
        else None
    ),
    country_name_col=(
        "country_norm"
        if "country_norm" in df_clean.columns
        else None
    ),
    clean_output_prefix="affiliation_clean",
    output_ror_col="ror_after_api",
    cache_file=CACHE_FILE,

    min_request_interval=0.25,
)

print(
    "ROR Client ID:",
    "vorhanden" if processor.client_id else "nicht vorhanden",
)
print(
    "Request-Abstand:",
    f"{processor.min_request_interval:.2f} Sekunden",
)
print(
    "Zeilen für den möglichen API-Lauf:",
    f"{len(df_clean):,}",
)

## 12. Optional: Exact-Fallback-Kandidaten ansehen

`generate_exact_candidates()` ruft **keine API** auf.

Die Funktion zeigt nur, welche Organisationsstrings der Processor bei einem Fall ohne brauchbaren `chosen=true`-Treffer als Exact-Fallback versuchen würde.

Das ist vor allem für Debugging nützlich.

`limit=3` bedeutet hier nur, dass maximal drei Kandidaten angezeigt werden.

In [ ]:
candidate_preview = df_clean[
    ["affiliation_clean"]
].sample(
    n=min(15, len(df_clean)),
    random_state=RANDOM_STATE,
).copy()

candidate_preview["exact_candidates"] = (
    candidate_preview["affiliation_clean"]
    .map(
        lambda value: processor.generate_exact_candidates(
            value,
            limit=3,
        )
    )
)

display(candidate_preview)

## 13. ROR API ausführen – `processor.process()`

Der normale vollständige Lauf ist:

```python
df_ror = processor.process(
    df_clean,
    checkpoint_file=CHECKPOINT_FILE,
    show_progress=True,
)
```

Wenn `max_rows` nicht angegeben wird, werden **alle offenen Zeilen** verarbeitet.

### Wichtige optionale Parameter

- `max_rows=10` – nur für kleine Tests; im Produktivlauf weglassen.
- `statuses=[...]` – nur bestimmte Status erneut bearbeiten.
- `force=True` – Statusfilter ignorieren und ausgewählte Zeilen erneut bearbeiten.
- `bypass_cache=True` – Cache für Requests bewusst umgehen.
- `checkpoint_file=...` – Zwischenstand regelmäßig speichern.
- `show_progress=True` – Fortschrittsbalken anzeigen.

Normaler Wiederanlauf nach einem abgebrochenen Lauf:

```python
processor.process(df, checkpoint_file=..., show_progress=True)
```

Der Cache verhindert dabei unnötige identische Requests.

`bypass_cache=True` sollte nur verwendet werden, wenn wirklich frische API-Antworten gewünscht werden.

In [ ]:
if RUN_ROR_API:
    df_ror = processor.process(
        df_clean,
        checkpoint_file=CHECKPOINT_FILE,
        show_progress=True,
    )
else:
    df_ror = df_clean.copy()
    print("ROR API wurde übersprungen.")

## 14. Was erzeugt der ROR-Processor?

Typische neue Felder:

- `affiliation_clean_1`, `affiliation_clean_2`, ... – strukturierte Organisationsteile.
- `ror_after_api` – neu ermittelter ROR.
- `ror_status` – z. B. `active` oder `inactive`.
- `status` – grobe Entscheidung der Pipeline.
- `status_detail` – genauere Begründung.
- `ror_successor_id`, `ror_successor_name` – bei historischen/inaktiven Records, wenn ein Successor bekannt ist.

Für einen sicheren aktiven ROR-Match wird der aktuelle ROR-Hauptname in `affiliation_clean_1` geschrieben.

Beispiel:

```text
Land and Water, CSIRO
```

kann zu

```text
affiliation_clean_1 = Commonwealth Scientific and Industrial Research Organisation
affiliation_clean_2 = Land and Water
```

werden.

`affiliation_clean` selbst bleibt als vorheriger Bereinigungsschritt erhalten.

In [ ]:
if RUN_ROR_API:
    clean_split_columns = sorted(
        [
            column
            for column in df_ror.columns
            if column.startswith("affiliation_clean_")
            and column[len("affiliation_clean_"):].isdigit()
        ],
        key=lambda column: int(
            column[len("affiliation_clean_"):]
        ),
    )

    result_columns = [
        column
        for column in [
            "id",
            "affiliation_part_raw",
            "affiliation_clean",
            *clean_split_columns,
            ORIGINAL_ROR_COLUMN,
            "ror_after_api",
            "ror_status",
            "status",
            "status_detail",
            "ror_successor_id",
            "ror_successor_name",
        ]
        if column in df_ror.columns
    ]

    display(
        df_ror[result_columns].sample(
            n=min(25, len(df_ror)),
            random_state=RANDOM_STATE,
        )
    )

## 15. Status verstehen

Die Hauptstatus sind absichtlich klein gehalten:

### `matched`

Die Pipeline hat genügend Evidenz für eine automatische ROR-Zuordnung.

Häufige Details:

- `affiliation_chosen` – ROR Affiliation Search hat `chosen=true` geliefert und der Treffer war plausibel.
- `exact_name_fallback` – eindeutiger exakter Name wurde im Fallback gefunden.
- `exact_name_country_fallback` – exakter Name konnte zusätzlich über das Land disambiguiert werden.
- `alternative_name_match` – exakte ROR-Namensvariante/Alias wurde erkannt.
- `original_ror_validated` – vorhandener ROR konnte durch die aktuelle Evidenz bestätigt werden.

### `unmatched`

Es wurde kein ausreichend guter ROR gefunden.

Typisch:

- `no_exact_name_match`

Das ist nicht automatisch ein Problem. Viele Firmen oder kleinere Organisationen existieren schlicht nicht in ROR.

### `review`

Es gibt einen Grund, den Fall manuell anzusehen.

Typische Details:

- `chosen_name_mismatch` – ROR hat zwar `chosen=true` geliefert, der Organisationsname war aber nicht plausibel im Originaltext auffindbar.
- `multiple_organizations` – mehrere eigenständige Organisationen könnten im Eintrag stecken.
- `ambiguous_exact_name` – mehrere exakte ROR-Kandidaten sind möglich.
- `ror_not_unique` – API-ROR und vorhandener `ror_original` unterscheiden sich.
- `exact_inactive_match` – exakter Treffer auf einen inaktiven historischen ROR.

### `api_error` / `rate_limited`

Technischer Fehler bzw. Rate Limit. Diese Fälle können später erneut verarbeitet werden.

In [ ]:
if RUN_ROR_API:
    status_summary = (
        df_ror[
            ["status", "status_detail"]
        ]
        .value_counts(dropna=False)
        .rename("count")
        .reset_index()
    )

    display(status_summary)

## 16. Review-Fälle ansehen

Diese Fälle sind die wichtigsten Kandidaten für manuelle Prüfung.

Es ist **nicht** das Ziel, `review` vollständig zu vermeiden. Ein kleiner Review-Anteil ist sinnvoller als aggressive False Positives.

In [ ]:
if RUN_ROR_API:
    review_cases = df_ror.loc[
        df_ror["status"].eq("review"),
        result_columns,
    ]

    print(f"Review-Fälle: {len(review_cases):,}")
    display(review_cases)

## 17. Unmatched-Fälle ansehen

Bei `unmatched` sollte zuerst geprüft werden, ob die Organisation überhaupt in ROR existiert.

Besonders bei:

- GmbHs,
- kleinen Unternehmen,
- lokalen Einrichtungen,
- sehr neuen Organisationen,

kann ein fehlender ROR völlig korrekt sein.

In [ ]:
if RUN_ROR_API:
    unmatched_cases = df_ror.loc[
        df_ror["status"].eq("unmatched"),
        result_columns,
    ].copy()

    print(f"Unmatched-Fälle: {len(unmatched_cases):,}")
    display(unmatched_cases)

## 18. Technische Fehler gezielt herausfiltern

Falls ein langer API-Lauf unterbrochen wurde, sind diese Status besonders relevant:

- `api_error`
- `rate_limited`
- `pending`

Sie können später erneut verarbeitet werden, ohne alle erfolgreichen Zeilen neu anzufassen.

In [ ]:
if RUN_ROR_API:
    retry_cases = df_ror.loc[
        df_ror["status"].isin(
            ["pending", "api_error", "rate_limited"]
        ),
        result_columns,
    ]

    print(f"Erneut zu verarbeitende technische Fälle: {len(retry_cases):,}")
    display(retry_cases)

## 19. `ror_original` vergleichen – aber nicht als Ground Truth behandeln

Ein vorhandener `ror_original` ist sehr nützlich für Qualitätskontrolle, kann aber selbst falsch sein.

Deshalb:

- gleiche IDs → starkes Konsistenzsignal,
- unterschiedliche IDs → genauer ansehen,
- vorhandener Original-ROR → nicht blind übernehmen.

Außerdem müssen ROR-URLs vor einem Vergleich normalisiert werden, weil z. B.

```text
ror.org/04mte1k06
```

und

```text
https://ror.org/04mte1k06
```

dieselbe ROR-ID meinen.

In [ ]:
def normalize_ror_for_comparison(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if not text:
        return pd.NA

    text = text.removeprefix("https://")
    text = text.removeprefix("http://")
    text = text.removeprefix("www.")

    return text.casefold()


if (
    RUN_ROR_API
    and ORIGINAL_ROR_COLUMN
    and ORIGINAL_ROR_COLUMN in df_ror.columns
):
    comparison = df_ror[
        [
            column
            for column in [
                "id",
                "affiliation_part_raw",
                ORIGINAL_ROR_COLUMN,
                "ror_after_api",
                "status",
                "status_detail",
            ]
            if column in df_ror.columns
        ]
    ].copy()

    comparison["ror_original_norm"] = (
        comparison[ORIGINAL_ROR_COLUMN]
        .map(normalize_ror_for_comparison)
    )
    comparison["ror_after_api_norm"] = (
        comparison["ror_after_api"]
        .map(normalize_ror_for_comparison)
    )

    both_present = comparison.dropna(
        subset=[
            "ror_original_norm",
            "ror_after_api_norm",
        ]
    )

    same = (
        both_present["ror_original_norm"]
        == both_present["ror_after_api_norm"]
    )

    print("Beide ROR vorhanden:", len(both_present))
    print("Davon gleiche ROR:", int(same.sum()))
    print("Davon unterschiedliche ROR:", int((~same).sum()))

    display(
        both_present.loc[
            ~same
        ]
    )
else:
    print("Keine Original-ROR-Spalte für einen Vergleich vorhanden.")

## 20. Ergebnis speichern – `save_csv()`

`save_csv()` speichert das DataFrame als CSV.

Im normalen Projektaufbau wird in den `data/`-Ordner geschrieben.

Beispiel:

```python
save_csv(
    df_ror,
    "affiliation_result.csv",
    overwrite=True,
)
```

`overwrite=True` ist bewusst explizit. Ohne diese Freigabe soll eine bestehende Ergebnisdatei nicht versehentlich überschrieben werden.

In [ ]:
save_csv(
    df_ror,
    OUTPUT_FILENAME,
    overwrite=True,
)

# Häufige Varianten

## A. Eigene CSV verwenden

Nur oben ändern:

```python
DATA_FILENAME = "meine_datei.csv"
SOURCE_AFFILIATION_COLUMN = "organization"
```

Die Datei vorher nach `data/` kopieren.

---

## B. Keine bestehende ROR-Spalte

```python
ORIGINAL_ROR_COLUMN = None
```

Der ROR-Processor funktioniert trotzdem normal.

---

## C. Bereits eine Affiliation pro Zeile

```python
SPLIT_AFFILIATIONS = False
```

Dann wird kein Semikolon-Split durchgeführt und der vorhandene Affiliation-Text direkt nach `affiliation_part_raw` kopiert.

---

## D. Andere Trennung innerhalb einer Zelle

Beispiel:

```python
AFFILIATION_DELIMITER = "|||"
```

`split_column()` behandelt den Delimiter literal, nicht als Regular Expression.

---

## E. Nur Location Cleaning testen

```python
RUN_ROR_API = False
```

Dann entstehen keine API-Kosten bzw. Wartezeiten.

---

## F. Nur wenige Zeilen mit ROR testen

Im `process()`-Aufruf temporär:

```python
max_rows=10
```

Für den vollständigen Lauf wieder entfernen bzw. `None` verwenden.

---

## G. API-Lauf bewusst komplett neu starten

1. Cache-Datei löschen oder einen neuen Cache-Dateinamen setzen.
2. Gegebenenfalls Checkpoint löschen.
3. Danach neu ausführen.

`bypass_cache=True` ist ebenfalls möglich, sollte aber nicht standardmäßig verwendet werden.

---

## H. Nach einem technischen Fehler weiterarbeiten

Normalerweise genügt ein erneuter Aufruf von `process()`.

Standardmäßig werden offene bzw. technische Status wie `pending`, `api_error` und `rate_limited` erneut verarbeitet.

---

## I. Zusätzliche Länderaliase

Beim Laden der Location-Ressourcen:

```python
resources = load_location_resources(
    additional_country_aliases={
        "mein alias": "AT",
    }
)
```

Der Wert rechts ist ein ISO-2-Ländercode.

---

## J. Weniger Location-Ausgabespalten

Es müssen nicht alle sechs Location-Spalten erzeugt werden:

```python
df_clean = process_locations(
    df_parts,
    column="affiliation_part_raw",
    resources=resources,
    output_columns=[
        "city_norm",
        "country_norm",
    ],
)
```

`affiliation_clean` wird trotzdem erzeugt.

# Quick Reference der wichtigsten Funktionen

| Funktion | Zweck | Zwingend | Häufig optional |
|---|---|---|---|
| `load_csv()` | CSV laden | Dateipfad | `sep`, `encoding`, weitere pandas-Optionen |
| `preview_dataframe()` | Spalten + Stichprobe ansehen | DataFrame | `n`, `random_state` |
| `rename_columns()` | Spalten umbenennen | DataFrame, Mapping | – |
| `split_column()` | in Zeilen oder Spalten splitten | `column`, `delimiter`, `into` | `output_column`, `output_prefix`, `row_id_column`, `drop_empty` |
| `load_location_resources()` | Länder/GeoNames laden | – | Pfade, `locales`, Aliase, Progress |
| `norm_country()` | einzelnes Land normalisieren | Wert, `resources` | – |
| `norm_city()` | einzelne Stadt normalisieren | Wert, `resources` | – |
| `process_locations()` | Orte extrahieren, `affiliation_clean` erzeugen | DataFrame, `column`, `resources` | Output-Spalten, Diagnostics, Address Removal |
| `RORAffiliationProcessor(...)` | ROR-Workflow konfigurieren | praktisch `api_input_col`, `clean_col` | Original-ROR, Länderfelder, Cache, Retry/Rate-Einstellungen |
| `generate_exact_candidates()` | Fallback-Kandidaten ohne API ansehen | Text | `limit` |
| `processor.process()` | ROR-Abfragen ausführen | DataFrame | `statuses`, `max_rows`, `force`, `bypass_cache`, Checkpoint, Progress |
| `save_csv()` | CSV speichern | DataFrame, Dateiname | `overwrite`, CSV-Optionen |

Die internen Hilfsfunktionen in `locations.py` und `ror_api.py` sind für den normalen Notebook-Workflow nicht notwendig. Für die tägliche Anwendung sollten möglichst nur die hier beschriebenen öffentlichen Funktionen verwendet werden.

# Empfohlener Arbeitsablauf

Für neue Daten würde ich praktisch immer so vorgehen:

1. CSV in `data/` kopieren.
2. Konfigurationszelle anpassen.
3. Laden und 5–10 Zeilen kontrollieren.
4. Affiliation-Split kontrollieren.
5. Location Cleaning kontrollieren.
6. Optional zuerst `max_rows=10` für einen kleinen API-Test verwenden.
7. Danach vollständigen API-Lauf starten.
8. `status` / `status_detail` ansehen.
9. `review` und relevante `unmatched` manuell prüfen.
10. Ergebnis speichern.

Nicht für jeden einzelnen Sonderfall neue Regeln in die Pipeline einbauen. Eine einfache, stabile Logik mit wenigen manuellen Review-Fällen ist für heterogene Affiliation-Daten meist sinnvoller als ein immer komplexeres Regelwerk.

# Kurzversion für den späteren Alltag

Wenn das Tutorial einmal bekannt ist, reduziert sich der normale Workflow im Wesentlichen auf diese Schritte:

In [ ]:
# 1. Laden
df = load_csv(PROJECT_ROOT / "data" / DATA_FILENAME)

# 2. Spaltenname anpassen
if SOURCE_AFFILIATION_COLUMN != STANDARD_AFFILIATION_COLUMN:
    df = rename_columns(
        df,
        {SOURCE_AFFILIATION_COLUMN: STANDARD_AFFILIATION_COLUMN},
    )

# 3. Mehrere Affiliations in einzelne Zeilen
df_parts = split_column(
    df,
    column=STANDARD_AFFILIATION_COLUMN,
    delimiter=AFFILIATION_DELIMITER,
    into="rows",
    output_column="affiliation_part_raw",
)

# 4. Locations
resources = load_location_resources()

df_clean = process_locations(
    df_parts,
    column="affiliation_part_raw",
    resources=resources,
    include_diagnostics=True,
)

# 5. ROR
processor = RORAffiliationProcessor(
    api_input_col="affiliation_part_raw",
    clean_col="affiliation_clean",
    original_ror_col=(
        ORIGINAL_ROR_COLUMN
        if ORIGINAL_ROR_COLUMN in df_clean.columns
        else None
    ),
    country_code_col="country_iso2",
    country_name_col="country_norm",
    min_request_interval=0.25,
)

df_ror = processor.process(
    df_clean,
    checkpoint_file=CHECKPOINT_FILE,
    show_progress=True,
)

# 6. Speichern
save_csv(
    df_ror,
    OUTPUT_FILENAME,
    overwrite=True,
)